# Section 1: Setting up the Environment

In [3]:
import os
from dotenv import load_dotenv

load_dotenv("api.env")

LANGCHAIN_KEY = os.getenv("LANGCHAIN_API_KEY")
HUGGINGFACE_API = os.getenv("HUGGINGFACEHUB_API_TOKEN")

# Section 2: Loading Documents

(a) Loading documents into a global variable

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

# For the purpose of testing, this version uses a pdf loader
loader = PyPDFDirectoryLoader("/content/RAG tester")
documents = loader.load()

(b) Splitting documents into manageable chunks

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100) # These are hyperparameters, can attempt tuning this using bayesian optimisation in the future
texts = splitter.split_documents(documents)

# Section 3: Indexing

(a) Embedding Text Chunks into the vector store

In [ ]:
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(texts, embeddings)

<ipython-input-4-1aa4911713de>:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or da

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(c) Declaring the number of chunks used in producing response

In [ ]:
k = 5
retriever = vectorstore.as_retriever(search_kwargs={"k": k})

# Section 4: Setting up the Generator

In [ ]:
from huggingface_hub import InferenceClient
client = InferenceClient(model="mistralai/Mistral-7B-Instruct-v0.3", token = os.environ['HUGGINGFACEHUB_API_TOKEN'])

from langchain_core.runnables import Runnable

class HuggingFaceChatRunnable(Runnable):
    def __init__(self, client, prompt_template, temperature, max_tokens):
        self.client = client
        self.prompt_template = prompt_template
        self.temperature = temperature
        self.max_tokens = max_tokens

    def invoke(self, inputs: dict, config: dict = None) -> str:
        prompt_str = self.prompt_template.format(**inputs)

        response = self.client.chat_completion(
            messages=[
                {"role": "user", "content": prompt_str}
            ],
            temperature=self.temperature,
            max_tokens=self.max_tokens
        )
        return response.choices[0].message["content"]

# Section 5: Routing Prompts

(a) Produce some fewshot examples to incorporate into routing prompt

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import FewShotChatMessagePromptTemplate

# Some fewshot examples
examples =[
    {
        "input": "What is creatine?",
        "output": "DEFINE",
    },
    {
        "input": "Why do athletes take protein after workouts?",
        "output": "EXPLAIN",
    },
    {
        "input": "How do I calculate my calorie needs?",
        "output": "PROCEDURE",
    },
    {
        "input": "Should I take whey or casein protein?",
        "output": "COMPARISON",
    },
    {
        "input": "What is the best way to embark on my weight loss journey?",
        "output": "ADVICE",
    },
]


example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_examples = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)


(b) Setting up the intent prompt

In [ ]:
intent_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an intent classifier for the field of interest in the query.
Given a question, classify it into one of the following intents:
- DEFINE: Asking for a definition or description
- EXPLAIN: Asking for reasoning or why something is the case
- PROCEDURE: Asking for how-to or steps
- ADVICE: Asking for personalized or practical suggestions
- COMPARISON: Asking to compare options
- GENERAL: Anything else
Return only the intent, nothing else.
Here are a few examples:""",
        ),
        # few shot examples
        few_shot_examples,
        # New question
        ("user", "{question}"),
    ]
)

(c) Set up routing

In [ ]:
router = HuggingFaceChatRunnable(client, intent_prompt, 0.0, 10)

# Section 6: Step Back Translation

(a) Step-back prompt

In [ ]:
# This are examples that shows the LLM what it is achieving through stepback

examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]

# Now translate this into an example_prompt
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

step_back_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot,
        # New question
        ("user", "Intent: {intent}\nQuestion: {question}"),
    ]
)

(c) Set up Step Back

In [ ]:
stepback = HuggingFaceChatRunnable(client, step_back_prompt, 0.0, 1024)

# Section 7: Routing Chains

(a) Chains

In [ ]:
#define prompt
defineprompt = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.
You are responding to a query with the intent: DEFINE.
Your answer should be:
- Comprehensive, but concise (1–3 sentences max)
- Factually correct and aligned with the provided context
- Free of speculation, advice, or subjective judgment
- Focused only on essential information—no unnecessary background or examples unless they resolve ambiguity
- Adjusted for multiple meanings if applicable
- Written in terminology appropriate to the user's domain or field


# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""

#explain prompt
explainprompt = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.
You are responding to a query with the intent: EXPLAIN.
Your answer should be:
- Clear and logically structured
- Focused on cause, reasoning, background, or significance
- Factually correct and aligned with the provided context
- Neutral in tone—avoid persuasion, speculation, or personal opinions
- Examples are welcome from the context provided, if it helps to improve understanding.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""

#procedure prompt
procedureprompt = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.
You are responding to a query with the intent: PROCEDURE.
Your answer should be:
- Structured as a clear, ordered list of steps (e.g., 1, 2, 3...)
- Focused on how-to instructions or best-practice sequences
- Specific, practical, and applicable to the user’s likely context
- Factually accurate and based on reliable knowledge
- Aligned with the provided context; ignore context if irrelevant

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:
1.
2.
3."""

#advice prompt
adviceprompt = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.
You are responding to a query with the intent: ADVICE.
Your answer should be:
- Actionable and practical, tailored to a general user (not personalized)
- Fact-based, but sensitive to nuance, caution, or best practices
- Free from subjective judgment or emotional language
- Respectful of varying conditions or assumptions
- Aligned with the provided context; if not relevant, ignore the context

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""

#comparison
comparisonprompt = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.
You are responding to a query with the intent: COMPARISON.
Your answer should be:
- A neutral, side-by-side analysis of options or alternatives
- Factually grounded—avoid personal recommendations unless one option is clearly superior based on evidence
- Clearly structured with bullet points or short paragraphs
- Helpful in illustrating pros and cons, similarities, and differences
- Consistent with the context provided; ignore it if irrelevant

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:
Option A:
Option B: """

#general prompt
generalprompt = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.
You are responding to a query with the intent: GENERAL.
Your answer should be:
- Informative and contextually aware
- Concise but flexible in length (aim for clarity)
- Objective and based on verifiable information
- Avoid speculation or personal opinion
- Aligned with the provided context if relevant

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""


(b) load prompt into a global variable

In [ ]:
from langchain_core.runnables import RunnableLambda

intent_router = RunnableLambda(lambda x: {
    "DEFINE": HuggingFaceChatRunnable(client, defineprompt, 0.0, 1024),
    "EXPLAIN": HuggingFaceChatRunnable(client, explainprompt, 0.0, 1024),
    "PROCEDURE": HuggingFaceChatRunnable(client, procedureprompt, 0.0, 1024),
    "ADVICE": HuggingFaceChatRunnable(client, adviceprompt, 0.0, 1024),
    "COMPARISON": HuggingFaceChatRunnable(client, comparisonprompt, 0.0, 1024),
    "GENERAL": HuggingFaceChatRunnable(client, generalprompt, 0.0, 1024),
}[x["intent"].strip()]
)

(c) Chain

In [ ]:
from langchain_core.runnables import RunnableMap, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

chain = (
    RunnableMap({
        "question": lambda x: x["question"],
        "step_back_question": lambda x: x["question"],
        "intent": lambda x: router.invoke({"question": x["question"]})
    })
    | RunnableLambda(lambda x: {
        "normal_context": retriever.invoke(x["question"]),
        "step_back_q": stepback.invoke({"intent" : x["intent"],"question": x["step_back_question"]}),
        "question": x["question"],
        "intent": next(iter(x["intent"])) if isinstance(x["intent"], set) else x["intent"]
    })
    | RunnableLambda(lambda x: {
        "step_back_context": retriever.invoke(x["step_back_q"]),
        "normal_context": x["normal_context"],
        "question": x["question"],
        "intent": x["intent"]
    })
    | intent_router
    | StrOutputParser()
)

# Section 8: Running the Query




In [ ]:
#question = input("Ask me anything! \n")

# Generate the Response
#response = chain.invoke({"question": question})
#print(response)

# Section 9: Creating a Web API

(a) Installing packages

In [ ]:
!pip install flask flask-cors pyngrok
!ngrok config add-authtoken 2xfZdlDmIgVQGcUtQZ9xvEqniPu_47KTWYCNPectJTS6LmjQD

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


(b) Define RAG function

In [ ]:
import logging
import time
import traceback
from huggingface_hub.utils import HfHubHTTPError

def RAG_pipeline(question, retries=3, delay=5):

    if not question or not isinstance(question, (str, dict)):
        return {"output": "⚠️ Invalid input. Please provide a valid question."}

    for attempt in range(retries):
        try:
            # Ensure input is in expected dict format
            query_input = {"question": question} if isinstance(question, str) else question

            response = chain.invoke(query_input)

            if not response:
                raise ValueError("No response received from the chain.")

            print(response)
            return response

        except HfHubHTTPError as e:
            logging.warning(f"Attempt {attempt + 1}: Hugging Face API error: {e}. Retrying in {delay} seconds...")
            time.sleep(delay)

        except KeyError as e:
            logging.error(f"KeyError: Missing key in response. Details: {e}")
            return {"output": "⚠️ Internal error: Invalid response format."}

        except ValueError as e:
            logging.error(f"ValueError: {e}")
            return {"output": f"⚠️ {str(e)}"}

        except Exception as e:
            logging.error("Unhandled Exception: %s", traceback.format_exc())
            break  # Do not retry on unexpected errors

    return {"output": "⚠️ Failed to retrieve a response after several attempts. Please try again later."}


(d) Build the Flask API

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import logging
import traceback

app = Flask(__name__)
CORS(app)  # Allow cross-origin requests

@app.route("/ask", methods=["POST"])
def ask():
    try:
        data = request.get_json(force=True)
        if not data or "question" not in data:
            return jsonify({"error": "⚠️ Missing 'question' field in request body."}), 400

        question = data["question"]

        if not isinstance(question, str) or not question.strip():
            return jsonify({"error": "⚠️ Invalid question format. Must be a non-empty string."}), 400

        # Call RAG pipeline
        answer = RAG_pipeline({"question": question})
        return jsonify({"answer": answer})

    except Exception as e:
        logging.error("Server Error: %s", traceback.format_exc())
        return jsonify({"error": "⚠️ Internal server error. Please try again later."}), 500

(e) Expose the RAG pipeline to the internet

In [ ]:
if __name__ == "__main__":
    # Start ngrok tunnel
    public_url = ngrok.connect(5000)
    print("Public URL:", public_url)

    # Run the Flask server
    app.run(port=5000)

Public URL: NgrokTunnel: "https://80a1-34-85-191-236.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 07:28:36] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 07:28:37] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 07:28:41] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 07:29:18] "POST /ask HTTP/1.1" 200 -


The benefits of studying Behavioral Sciences, as outlined in the provided context, include:

1. Enhanced understanding of human behavior: By studying Behavioral Sciences, one can gain a deeper insight into how people make decisions, how they perceive and process information, and how they interact with each other and their environment.

2. Improved accuracy of predictions: The theories and models developed in Behavioral Sciences can help make more accurate predictions about human behavior, which is particularly useful in fields such as economics, marketing, and psychology.

3. Increased interest and fun: Behavioral Sciences, often referred to as the "un-dismal" science, is more engaging and enjoyable compared to traditional sciences. It offers a unique blend of psychology, economics, and other disciplines, making it a fascinating field of study.

4. Contribution to making the world a better place: By applying the wisdom gained from Behavioral Sciences, we can design interventions and po

INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 07:30:10] "OPTIONS /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 07:30:18] "POST /ask HTTP/1.1" 200 -


Behavioral Economics is an interdisciplinary field that combines economic theory with insights from psychology and other social sciences to better understand and predict human behavior. It aims to improve the accuracy of economic predictions by incorporating the irrationalities, biases, and heuristics that often influence decision-making.


INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 07:32:03] "OPTIONS /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 07:32:13] "POST /ask HTTP/1.1" 200 -


Behavioral sciences can be used in a harmful manner when individuals or organizations with bad intentions exploit the findings of these sciences for self-serving purposes, often at the expense of the individuals who have been influenced by these interventions. This misuse can occur in various contexts, such as businesses or governments, and it can lead to unethical practices that manipulate people's decisions and behaviors without their full understanding or consent. It is essential to carefully select and rigorously test nudges and interventions based on behavioral science findings to minimize the potential for harm and ensure that they are used for the greater good.
